FACE LANDMARKS DETECTION: GEOMETRICA DEL VOLTO E ANALISI COMPORTAMENTALE

Immagina la differenza tra vedere la sagoma di una persona vista a distanza e poter leggere l'espressione degli occhi da vicino.

I Landmarks è la capacità di mappare la geometria profonda del viso per capire non solo dove si trova ma cosa sta comunicando.
Quindi capiamo come leggere un volto

- Mappatura del volto: i 68 punti caratteristici del modello Dlib e la dense mash di MediaPipe
- Analisi dello stato oculare: calcolo dell'Eye Aspect Ratio (EAR) per rilevamento delle stanchezza
- Normalizzazione e pre-processing: tecniche di allineamento del volto per ottimizzare il riconoscimento.

La Face Landmarks Detection è il passo successivo rispetto alla semplice face detection
La Face Detection dice:
    "qui c'è un volto"
La Face Landmarks Deteciont dice:
    "questi sono i punti geometrici significativi del volto"
esempio: occhio destra, occhio sinistro, punta del naso, angoli bocca, ecc
In pratica trasforma il volto in una struttura geometrica di punti.

Mappatura dei Landmarks Facciali

La rilevazione dei landmarks consiste nell'individuazione automatica di punti specifici che definiscono la struttura anatomica del volto. A differenza della semplice bounding box  (che è come un recinto che contiene tutta la casa), questa tecnica ci permette di comprendere l'orientamento e l'espressione del soggetto.
Landmarks sono gli angoli degli occhi, la punta del naso, il profilo delle labbra, ecc 
Questo permete di superare la statisticità, se io inclino la testa la bounding box rimane un rettangolo ma i lenadmarks si muovono con la testa.
Questi punti fungono da ancora geometrica per compiti complessi come il face swapping, l'analisi micro-espressioni e la stima della posa della testa rispetto alla camera.

Ma quanti di questi punti ci servono per capire un volto?

Modelli e Standard di Riferimento
Dal modello a 68 punti alla mesh densa
Storicamente esistono modelli con pochi landkmark, per esempio 5 o 68 punti. 
I modelli a 68 punti sono stati utilizzati per anni come standard
Oggi abbiamo modelli come MediaPipe che invece di 68 punti ne generarno 400 ed oltre, creando una vera e propria maschera 3D, una mash che aderisce al volto come una maschera di lattice.
Esempio: Face Mash di Google, per esempio, utilizza rappresentazioni 3D dense di 468 vertici.

Con Dlib, il modello classico, restituisce 68 punti caratteristici del volto
Dlib descrive questi Landmarks come punti su occhi, sopracciglia, naso, ecc. Per ogni punto sono indicati x e y, una faccia diventa quindi:
[(x0,y0),(x1,y1),(x2,y2),...,(x67,y67)]
Non è più solo: "volto trovato" ma una struttura geometrica

Andiamo a vedere come questi punti si aggregano per formare l'anatomia facciale
Dlib è quindi una rappresentazione relativamente sparsa di 68 punti
MediaPipe Face Mesh invece produce una mesh 3D facciale di 468 vertici, abbastanza fitta da descrivere dettagliatamente superficie, occhim labbre, ecc

Anatomia dei Landmarks
I Landmarks non capiscono il volto, forniscono geometrie affidabili su cui poi costruirsi misure e trasformazioni.
Ognu gruppo di punti ha una missione
- Landmarks Oculari e Labbiali: vengono utilizzaati per monitorare il battito delle ciglia o il movimento della bocca, fondamentali per applicazioni di interfaccia uomo-macchina. Ci possono dire se stiamo parlando, ridendo o se stiamo per addormentarci.
- Profilo della Mascella: permette di definire il contorno del volto e supporta gli algoritmi di stima della rotazione (yaw, pitch, roll). Ci dice come la testa è orientata nello spazio.
- Stabilità Temporale: l'uso di filtri come One Euro Filter riduce lo sfarfalio dei punti tra un frame e l'altro migliorando l'esperienza utente.

Ma come facciamo a misurare la distanza tra questi punti, in modo che abbia senso matematio?

Metrica della Distanza tra Punti
Calcolo della similarità geometrica
Se mi avvicino alla webcam, ma distanza, in pixel, dei miei occhi aumenta.
La mia faccia, è sempre la mia faccia, per questo non possiamo usare misure assolute.
Usiamo la distanza euclidea ma dobbiamo normalizzarla.
Prendiamo una misura fissa (es. la distanza tra le pupille) e la usiamo come unità di misura fondamentale per questo volto.
In questo modo, che io sia a 10 mm dalla fotocamere o a 3 mt il rapporto tra la distanza delle mie caratteristiche resterà costante.

Ma parlando di rapporti costanti, vediamo come questo concetto di permette di 'salvare delle vite'

Eye Aspect Ratio (EAR)
Rilevamento del battito e della fatica
Come facciamo a capire se qualcuno è stanco?
Invece di cerca di capirlo con una rete neurale complessa, utilizziamo la geometria, ci serve capire quanto l'occhio è schiacciato rispetto alla sua larghezza.
L'Eye Aspect Ragio è un parametri numerico che descrive lo stato di apertura dell'occhio. 
Questa tecnica ha rivoluzionato i sistemi di sicurezza stradale, permettendo di rilevare il colpo di sonno tramite semplici webcam.
Invece di addestrare un classificatore per 'occhio aperto' o 'occhio chiuso', utilizziamo una formula geometrica derivata dai landmarks che fornisce un segnale continuo e affidabile.

Ma come si comporta l'EAR mentre sbattiamo le ciglia?

Dinamica del Segnale EAR
Iterpretazione dei dati oculari.
Quando l'occhio è aperto EAR è alto e stabile, quando sbattiamo le ciglia l'EAR crolla, ma solo per una frazione di secondo, poi risale, questo è normale.
Ma se il segnale resta basso per 1, 2 secondi, abbiamo un problema, quello è un segnale di allarme.

Parametrizzazione dell'Algoritmo 
Per calcolare l'EAR usiamo 6 punti per occhio, questo perchè non tutti gli occhi sono uguali.
Questi parametri vanno calibrati per ogni persona. In un sistema a guida autonoma vanno calibrati i nei primi secondi di guida, in modo da adattarsi a quel guidatore specifico.
- Identicazione dei 6 Punti: l'occhio viene descritto da 6 punti specifici: due per gli angoli, e quattro per le palpebre superiori e inferiori
- Normalizzazione individuale: ogni utente ha un EAR a riposo differente; è buona norma calibrare il sistema nei primi secondi di  utilizzo
- Gestione dei Falsi Positivi: occhiali da vista, o condizoni di luce scarsa, possono degradare la precisione dei landkmarks, richiedendo l'uso di modelli di deep learning più robusti.

Formula che traduce questi punti in un numero utile
Formalizzazione dell'EAR
L'equiazione della sonnolenza
L'algoritmo calcola la media delle distanza verticali divisa per la distanza orizzontale. Questo rapporto rimane costante indipendentemente dalla distanza del volta dal sensore, rendendolo estremamente robusto.
I punti p2, p6, p3, p5 rappresentano le altezze verticali, mentre p1, p4 definiscono la larghezza dell'occhio.
Nell'EAR se raddoppia la larghezza (perchè ci avviciniamo alla camera) raddoppia anche l'altezza, ed il risultato finale dell'EAR rimane identico.

Ma oltre a monitorare la stanchezza i Landmarks ci servono anche per fare pulizia dei nostri dati

Allineamento e Frontalizzazione
Standarizzare l'input per il riconoscimento
Il riconoscimento facciale fallisce spesso a causa di variazioni di posa. 
Il riconoscimento facciale odia le inclinazioni
Se un volto è inclinato, le feature estratte dalla rete neurale saranno distore rispetto a quelle presenti nel database. Una rete neurale vedo una faccia storta come un oggetto completamente diverso.
L'allineamento o frontalizzazione, è l'arte di prendere quella foto storta e raddrizzarla virtualmente, come se stessimo facendo una fototessera perfetta in ogni momento.
L'allineamento utilizza i landmarks per ruotare, scalare e centrare il volto in  un formato standard, processo noto come frontalizzazione.

Trasformazioni Affini
Manipolazione geometrica dell'immagine
La trasformazione affine è come un foglio di gomma che possiamo tirare e allungare.
Usiamo gli occhi come la nostra ancora, si calcola l'angolo della retta che congiunge i centri degli occhi e si porta questa retta sempre parallela al suolo.
Poi scaliamo la faccia in modo che l'immagine occupi sempre lo stesso numero di pixel.
In questo modo, diamo al modello di riconoscimento, un input standarizzato.

Ma ne vale la pena spendere tempo in questo pre-processing?
Assolutamente si

Benefici del Pre-processing
- Riduzione della Varianza: eliminando la rotazione, il modello di riconoscimento può concentrarsi esclusivamente sui tratti somatici unici
- Efficienza computazionale: immagini allineate permettono l'uso di crop più piccoli, riducendo il numero di parametri da elaborare
- Invarianza alla Posa: sebbene non risolva occlusioni totali, l'allineamento mitiga drasticamente l'effetto di inclinazione lievi della testa.
a: occhi, naso, bocca, mandibola, ecc


In [ ]:
# python -m pip install --upgrade mediapipe
import os
import sys
import cv2
import numpy as np
import requests
from io import BytesIO
from PIL import Image
from typing import List, Tuple, Optional

# --- CONFIGURAZIONE BACKEND ---
# Keras 3 è un framework agnostico. Impostiamo "torch" (PyTorch) come motore di calcolo
# per sfruttare l'accelerazione hardware e l'interoperabilità con i modelli moderni.
os.environ["KERAS_BACKEND"] = "torch"
import keras

# --- NUOVA IMPORTAZIONE MEDIAPIPE TASKS ---
# MediaPipe si è evoluto verso le Tasks API, che utilizzano bundle di modelli (.task)
# per un'inferenza più efficiente e cross-platform.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

class FaceAnalysisEngine:
    """
    Advanced biometric engine based on the MediaPipe Tasks API.

    This class handles the end-to-end pipeline for facial analysis, including:
    - Automatic management of the .task model bundle (download and initialization).
    - 3D Face Landmarking with 478 points (Face Mesh).
    - Geometric normalization (Pose Alignment) via affine transformations.
    - Vigilance analysis using the Eye Aspect Ratio (EAR).
    """
    
    def __init__(self, model_path: str = "face_landmarker.task"):
        """
        Initializes the biometric detector.

        Args:
            model_path (str): Local path to the 'face_landmarker.task' bundle.
                             If missing, it will be downloaded automatically.
        
        The initialization configures the vision task options, sets confidence
        thresholds for detection and tracking, and prepares the 478-point mesh indices.
        """
        # URL ufficiale per scaricare il bundle del modello se mancante
        self.model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
        
        # Verifica se il modello esiste localmente, altrimenti lo scarica
        self._ensure_model_exists(model_path)

        # 1. BaseOptions: Configura le impostazioni di base come il percorso del modello AI (.task).
        # Il file .task contiene il modello TFLite e i metadati di pre/post-elaborazione.
        base_options = python.BaseOptions(model_asset_path=model_path)
        
        # 2. FaceLandmarkerOptions: Configura il comportamento specifico del rilevatore.
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            output_face_blendshapes=True, # Abilita il calcolo delle espressioni facciali (es. occhi chiusi)
            num_faces=1,                  # Limita l'analisi a un solo volto per ottimizzare le prestazioni
            min_face_detection_confidence=0.5, # Soglia minima per considerare rilevato un volto
            min_face_presence_confidence=0.5,  # Soglia minima per confermare la presenza del volto
            min_tracking_confidence=0.5        # Soglia minima per mantenere il tracciamento tra i frame
        )
        
        # Creazione del detector tramite le opzioni definite
        try:
            self.detector = vision.FaceLandmarker.create_from_options(options)
            print("MediaPipe Face Landmarker (Tasks API) inizializzato con successo.")
        except Exception as e:
            print(f"Errore critico durante l'inizializzazione: {e}")
            self.detector = None

        # --- INDICI MESH A 478 PUNTI ---
        # Indici delle palpebre per il calcolo dell'EAR (Eye Aspect Ratio)
        self.EYE_LEFT = [362, 385, 387, 263, 373, 380]
        self.EYE_RIGHT = [33, 160, 158, 133, 153, 144]
        
        # Indici dei centri delle iridi, utilizzati per l'allineamento della testa
        self.IRIS_LEFT = 468
        self.IRIS_RIGHT = 473

    def _ensure_model_exists(self, path: str):
        """
        Garantisce la presenza del bundle del modello richiesto nel filesystem locale.

        Se il file .task non viene trovato, avvia un download in streaming dai server ufficiali
        Google Cloud Storage. L'uso dello streaming garantisce l'efficienza della memoria
        anche per modelli di grandi dimensioni.

        Argomenti:
            path (str): Il percorso di destinazione previsto per il modello.
        """
        if not os.path.exists(path):
            print(f"Modello '{path}' non trovato. Download in corso dai server Google...")
            try:
                # Download a chunk per gestire file di grandi dimensioni senza saturare la RAM
                response = requests.get(self.model_url, stream=True, timeout=30)
                response.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print("Download completato correttamente.")
            except Exception as e:
                print(f"Errore durante il download del modello: {e}")
                sys.exit(1)

    def fetch_image(self, url: str) -> np.ndarray:
        """
        Scarica un'immagine remota e la prepara per l'elaborazione con OpenCV.

        Argomenti:
            url (str): L'URL dell'immagine da recuperare.

        Ritorna:
            np.ndarray: L'immagine in formato BGR (Standard OpenCV).
        
        Nota: L'immagine viene convertita dal formato RGB di PIL a NumPy/BGR poiché
        OpenCV utilizza l'ordine dei canali BGR.
        """
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        # OpenCV lavora in BGR, quindi invertiamo i canali
        return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

    def get_ear(self, landmarks, eye_indices: List[int], w: int, h: int) -> float:
        """
        Calcola l'Eye Aspect Ratio (EAR) per un singolo occhio.

        L'EAR è un valore scalare che correla con il livello di apertura dell'occhio.
        Formula: EAR = (||p2 - p6|| + ||p3 - p5||) / (2 * ||p1 - p4||)
        dove:
            - p1, p4: angoli orizzontali dell'occhio.
            - p2, p3, p5, p6: punti verticali della palpebra.

        Argomenti:
            landmarks: Lista dei landmark facciali normalizzati da MediaPipe.
            eye_indices (List[int]): Indici specifici della mesh per l'occhio.
            w (int): Larghezza dell'immagine.
            h (int): Altezza dell'immagine.

        Ritorna:
            float: Il valore EAR calcolato. Un valore basso indica un occhio chiuso.
        """
        # Convertiamo i landmark normalizzati (0-1) in coordinate pixel reali
        p = [np.array([landmarks[i].x * w, landmarks[i].y * h]) for i in eye_indices]
        
        # Distanze tra i punti superiori e inferiori della palpebra
        v1 = np.linalg.norm(p[1] - p[5])
        v2 = np.linalg.norm(p[2] - p[4])
        
        # Distanza tra gli angoli esterni dell'occhio
        horiz = np.linalg.norm(p[0] - p[3])
        
        # Calcolo finale del rapporto
        return (v1 + v2) / (2.0 * horiz) if horiz > 0 else 0.0

    def align_face(self, image: np.ndarray, landmarks) -> np.ndarray:
        """
        Esegue la normalizzazione della posa (Allineamento del volto).

        Questo metodo calcola l'angolo di inclinazione basato sui centri delle iridi
        e applica una rotazione affine per rendere l'asse orizzontale del volto 
        parallelo all'asse orizzontale del frame.

        Questo passaggio è CRUCIALE per la precisione dell'EAR in quanto elimina
        la distorsione verticale causata dall'inclinazione della testa.

        Argomenti:
            image (np.ndarray): Il frame BGR originale.
            landmarks: Landmark grezzi dal primo rilevamento.

        Ritorna:
            np.ndarray: L'immagine ruotata (allineata).
        """
        h, w = image.shape[:2]
        l_iris = landmarks[self.IRIS_LEFT]
        r_iris = landmarks[self.IRIS_RIGHT]
        
        # Posizione dei centri delle iridi in pixel
        l_center = (l_iris.x * w, l_iris.y * h)
        r_center = (r_iris.x * w, r_iris.y * h)
        
        # Calcolo dell'angolo di inclinazione tramite arcotangente
        angle = np.degrees(np.arctan2(r_center[1] - l_center[1], r_center[0] - l_center[0]))
        
        # Centro della rotazione (punto medio tra gli occhi)
        eye_mid = (int((l_center[0] + r_center[0]) / 2), int((l_center[1] + r_center[1]) / 2))
        
        # Creazione della matrice di rotazione e applicazione del warping
        M = cv2.getRotationMatrix2D(eye_mid, angle, 1.0)
        return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC)

    def analyze(self, image_url: str):
        """
        Esegue l'intera pipeline di analisi biometrica.

        Flusso di lavoro:
        1. Fetch: Scarica l'immagine dall'URL fornito.
        2. Rilevamento Iniziale: Trova il volto e i centri degli occhi nell'immagine grezza.
        3. Allineamento: Ruota l'immagine per orizzontalizzare lo sguardo.
        4. Analisi di Precisione: Esegue nuovamente il landmarker sull'immagine ALLINEATA per la massima precisione EAR.
        5. Rendering UI: Disegna i landmark e visualizza lo stato (Sveglio/Affaticato).

        Argomenti:
            image_url (str): URL di origine della foto.
        """
        # 1. Recupero dati
        img_bgr = self.fetch_image(image_url)
        h, w = img_bgr.shape[:2]
        
        # Conversione obbligatoria per MediaPipe Tasks API
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        
        # 2. Rilevamento iniziale
        detection_result = self.detector.detect(mp_image)
        if not detection_result.face_landmarks:
            print("Nessun volto rilevato nell'immagine.")
            return

        # Estraiamo i landmark del primo volto rilevato
        raw_lms = detection_result.face_landmarks[0]
        
        # 3. Normalizzazione geometrica del volto
        aligned_img = self.align_face(img_bgr, raw_lms)
        
        # 4. Analisi di precisione sull'immagine allineata
        mp_image_aligned = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(aligned_img, cv2.COLOR_BGR2RGB))
        res_aligned = self.detector.detect(mp_image_aligned)
        
        if res_aligned.face_landmarks:
            final_lms = res_aligned.face_landmarks[0]
            
            # Calcolo dell'EAR per entrambi gli occhi e media
            ear_l = self.get_ear(final_lms, self.EYE_LEFT, w, h)
            ear_r = self.get_ear(final_lms, self.EYE_RIGHT, w, h)
            avg_ear = (ear_l + ear_r) / 2.0
            
            # 5. Rendering della User Interface grafica
            self._draw_ui(aligned_img, final_lms, avg_ear)
            
            # Mostriamo l'anteprima e salviamo il file
            Image.fromarray(cv2.cvtColor(aligned_img, cv2.COLOR_BGR2RGB)).show()
            cv2.imwrite("aligned_face_2026.jpg", aligned_img)
            print(f"Analisi completata con successo. EAR calcolato: {avg_ear:.4f}")

    def _draw_ui(self, img, lms, ear):
        """
        Disegna gli elementi grafici sul frame.
        Include i punti della mesh e le etichette di testo dinamiche.
        """
        h, w = img.shape[:2]
        
        # Disegno di ogni punto della Face Mesh (punti verdi)
        for p in lms:
            cv2.circle(img, (int(p.x * w), int(p.y * h)), 1, (0, 255, 0), -1)
        
        # Logica di stato basata sulla soglia EAR (0.22 è un valore tipico per la sonnolenza)
        status = "ALERT (SVEGLIO)" if ear > 0.22 else "DROWSY (AFFATICATO)"
        color = (0, 255, 0) if ear > 0.22 else (0, 0, 255) # Verde se sveglio, Rosso se stanco
        
        # Stampa del testo informativo sull'immagine
        cv2.putText(img, f"EAR: {ear:.3f} | {status}", (20, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

# --- BLOCCO DI ESECUZIONE ---
if __name__ == "__main__":
    # Inizializziamo il motore (scatenerà il download se necessario)
    engine = FaceAnalysisEngine()
    
    # Se il motore è pronto, eseguiamo l'analisi su una foto di test
    if engine.detector:
        # Immagine di test: volto maschile in alta risoluzione
        TEST_URL = "https://images.pexels.com/photos/2379004/pexels-photo-2379004.jpeg"
        engine.analyze(TEST_URL)